# Project 11 - Dataset Cleaning and Split Preparation

This notebook is the second stage of the project workflow.

It starts from the four FakeNewsNet CSV files, performs deterministic cleaning, creates leakage-controlled train/validation/test splits, and optionally prepares valid-image-only and image-de-leaked split files when downloaded images are already available.

Run order:
1. `01_FakeNewsNet_EDA_FINAL_Enhanced.ipynb`
2. this cleaning notebook
3. image download/preparation notebook, if images are missing
4. multimodal model training notebook


## 1. Setup and Dataset Gate

In [ ]:
# Optional install command, run only if the environment is missing packages:
# !pip install pandas numpy scikit-learn pillow --quiet

import os
import re
import json
import hashlib
import warnings
import urllib.request
from pathlib import Path
from urllib.parse import urlparse

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 140)
pd.set_option('display.max_columns', 40)

SEED = 42
N_PER_SOURCE_LABEL = 5000
DOWNLOAD_DATASET_IF_MISSING = True
WRITE_OUTPUTS = True
RUN_IMAGE_HASH_DELEAKAGE = True

CSV_DIR = Path('dataset')
DATA_SPLIT_DIR = Path('data_splits')
IMAGE_DIR = Path('images')
MANIFEST_DIR = Path('image_manifests')
FINAL_ARTIFACT_DIR = Path('final_artifacts')

for path in [CSV_DIR, DATA_SPLIT_DIR, IMAGE_DIR, MANIFEST_DIR, FINAL_ARTIFACT_DIR]:
    path.mkdir(exist_ok=True)

REQUIRED_CSV_FILES = {
    'politifact_fake': CSV_DIR / 'politifact_fake.csv',
    'politifact_real': CSV_DIR / 'politifact_real.csv',
    'gossipcop_fake': CSV_DIR / 'gossipcop_fake.csv',
    'gossipcop_real': CSV_DIR / 'gossipcop_real.csv',
}

DATASET_URLS = {
    'politifact_fake': 'https://raw.githubusercontent.com/KaiDMML/FakeNewsNet/master/dataset/politifact_fake.csv',
    'politifact_real': 'https://raw.githubusercontent.com/KaiDMML/FakeNewsNet/master/dataset/politifact_real.csv',
    'gossipcop_fake': 'https://raw.githubusercontent.com/KaiDMML/FakeNewsNet/master/dataset/gossipcop_fake.csv',
    'gossipcop_real': 'https://raw.githubusercontent.com/KaiDMML/FakeNewsNet/master/dataset/gossipcop_real.csv',
}

def dataset_file_status():
    rows = []
    for name, path in REQUIRED_CSV_FILES.items():
        rows.append({
            'name': name,
            'path': str(path),
            'exists': path.exists(),
            'size_mb': round(path.stat().st_size / 1024**2, 2) if path.exists() else 0.0,
        })
    return pd.DataFrame(rows)

def ensure_minimal_csv_dataset(download_if_missing=True):
    status = dataset_file_status()
    missing = status.loc[~status['exists'], 'name'].tolist()
    if not missing:
        print('Dataset gate: all required CSV files already exist. Download skipped.')
        return status

    print('Dataset gate: missing CSV files:', missing)
    if not download_if_missing:
        raise FileNotFoundError(f'Missing required CSV files: {missing}')

    for name in missing:
        url = DATASET_URLS[name]
        target = REQUIRED_CSV_FILES[name]
        tmp_target = target.with_suffix(target.suffix + '.download')
        print(f'Downloading {name} -> {target}')
        try:
            urllib.request.urlretrieve(url, tmp_target)
            tmp_target.replace(target)
        except Exception as exc:
            if tmp_target.exists():
                tmp_target.unlink()
            raise RuntimeError(
                f'Could not download {name}. Check internet access or manually place the CSV in dataset/.'
            ) from exc

    return dataset_file_status()

dataset_status = ensure_minimal_csv_dataset(DOWNLOAD_DATASET_IF_MISSING)
display(dataset_status)

CSV_WRITE_LOG = []

def safe_to_csv(df, path, index=False, **kwargs):
    """Write CSV safely on Windows. If the target is locked, write a _new fallback file."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        df.to_csv(path, index=index, **kwargs)
        CSV_WRITE_LOG.append({'requested_path': str(path), 'written_path': str(path), 'status': 'ok'})
        return path
    except PermissionError:
        fallback = path.with_name(path.stem + '_new' + path.suffix)
        df.to_csv(fallback, index=index, **kwargs)
        CSV_WRITE_LOG.append({'requested_path': str(path), 'written_path': str(fallback), 'status': 'locked_fallback'})
        print(f'PermissionError: {path} is locked. Wrote fallback file instead: {fallback}')
        return fallback


## 2. Load Raw CSV Metadata

In [ ]:
def load_csv(source: str, label: str) -> pd.DataFrame:
    path = CSV_DIR / f'{source}_{label}.csv'
    if not path.exists():
        raise FileNotFoundError(f'Missing required CSV: {path}')

    df = pd.read_csv(path)
    required = {'id', 'news_url', 'title', 'tweet_ids'}
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(f'{path} is missing expected columns: {missing}')

    df['source'] = source
    df['label'] = label
    return df

raw_parts = [
    load_csv('politifact', 'fake'),
    load_csv('politifact', 'real'),
    load_csv('gossipcop', 'fake'),
    load_csv('gossipcop', 'real'),
]
raw = pd.concat(raw_parts, ignore_index=True)
raw['_row_uid'] = np.arange(len(raw))

raw_audit = raw.groupby(['source', 'label']).size().rename('n_rows').reset_index()
safe_to_csv(raw_audit, FINAL_ARTIFACT_DIR / 'cleaning_raw_source_label_counts.csv', index=False)

print(f'Raw rows: {len(raw):,}')
display(raw_audit)
display(raw.head(3))

## 3. Deterministic Text and Metadata Cleaning

In [ ]:
def clean_text_field(value):
    if pd.isna(value):
        return ''
    return re.sub(r'\s+', ' ', str(value)).strip()

def normalize_title_for_split(value):
    value = clean_text_field(value).lower()
    value = re.sub(r'[^a-z0-9]+', ' ', value)
    return re.sub(r'\s+', ' ', value).strip()

def extract_domain(url):
    url = clean_text_field(url)
    if not url:
        return ''
    parsed = urlparse(url if re.match(r'^https?://', url, flags=re.I) else 'http://' + url)
    return parsed.netloc.lower().replace('www.', '')

def count_tweet_ids(value):
    text = clean_text_field(value)
    if not text:
        return 0
    return len([x for x in re.split(r'[\s,\t]+', text) if x.strip()])

def punct_density(text):
    text = clean_text_field(text)
    if not text:
        return 0.0
    return sum(1 for ch in text if not ch.isalnum() and ch != ' ') / len(text)

clean = raw.copy()
for col in ['id', 'news_url', 'title', 'tweet_ids', 'source', 'label']:
    clean[col] = clean[col].map(clean_text_field)

removed_rows = []

empty_title_mask = clean['title'].eq('')
if empty_title_mask.any():
    removed_rows.append(clean.loc[empty_title_mask].assign(removal_reason='empty_or_missing_title'))
clean = clean.loc[~empty_title_mask].copy()

duplicate_id_mask = clean.duplicated(subset=['id', 'source'], keep='first')
if duplicate_id_mask.any():
    removed_rows.append(clean.loc[duplicate_id_mask].assign(removal_reason='duplicate_id_within_source'))
clean = clean.loc[~duplicate_id_mask].copy()

clean['title_len_chars'] = clean['title'].str.len()
clean['title_len_words'] = clean['title'].str.split().str.len()
clean['title_has_null'] = False
clean['title_is_empty'] = clean['title'].eq('')
clean['title_is_url'] = clean['title'].str.startswith(('http://', 'https://', 'www.'))
clean['title_all_caps'] = clean['title'].apply(lambda t: t.isupper() and len(t.split()) > 3)
clean['title_punct_density'] = clean['title'].apply(punct_density)
clean['title_exclaim_count'] = clean['title'].str.count('!')
clean['title_question_count'] = clean['title'].str.count(r'\?')
clean['url_missing'] = clean['news_url'].eq('')
clean['domain'] = clean['news_url'].apply(extract_domain)
clean['url_len'] = clean['news_url'].str.len()
clean['n_tweet_ids'] = clean['tweet_ids'].apply(count_tweet_ids)
clean['usable_text'] = clean['title_len_words'].ge(2)
clean['title_norm_split'] = clean['title'].apply(normalize_title_for_split)

removed = pd.concat(removed_rows, ignore_index=True) if removed_rows else pd.DataFrame(columns=list(raw.columns) + ['removal_reason'])

cleaning_audit = pd.DataFrame([
    {'stage': 'raw', 'n_rows': len(raw)},
    {'stage': 'removed_empty_or_missing_title', 'n_rows': int(empty_title_mask.sum())},
    {'stage': 'removed_duplicate_id_within_source', 'n_rows': int(duplicate_id_mask.sum())},
    {'stage': 'clean_full', 'n_rows': len(clean)},
])

if WRITE_OUTPUTS:
    safe_to_csv(clean, DATA_SPLIT_DIR / 'clean_full.csv', index=False)
    safe_to_csv(removed, FINAL_ARTIFACT_DIR / 'cleaning_removed_rows.csv', index=False)
    safe_to_csv(cleaning_audit, FINAL_ARTIFACT_DIR / 'cleaning_row_count_audit.csv', index=False)

print(f'Clean rows: {len(clean):,}')
display(cleaning_audit)
display(clean.groupby(['source', 'label']).size().unstack(fill_value=0))

## 4. Balanced, Title-De-Leaked Train/Validation/Test Splits

In [ ]:
pool = clean.copy()

blank_norm_mask = pool['title_norm_split'].eq('')
blank_norm_count = int(blank_norm_mask.sum())
pool = pool.loc[~blank_norm_mask].copy()

before_title_dedupe = len(pool)
pool = (
    pool.sort_values(['source', 'label', 'id'])
    .drop_duplicates(subset=['title_norm_split'], keep='first')
    .copy()
)
title_duplicate_rows_removed = before_title_dedupe - len(pool)

print(f'Removed blank normalized titles before split: {blank_norm_count:,}')
print(f'Removed duplicate normalized-title rows before split: {title_duplicate_rows_removed:,}')
display(pool.groupby(['source', 'label']).size().unstack(fill_value=0))

def balanced_sample_by_source_label(df, n_per_group, seed=42):
    parts = []
    rows = []
    for (source, label), group in df.groupby(['source', 'label']):
        n = min(n_per_group, len(group))
        sampled = group.sample(n=n, random_state=seed)
        parts.append(sampled)
        rows.append({'source': source, 'label': label, 'available': len(group), 'sampled': n})
    return pd.concat(parts, ignore_index=True), pd.DataFrame(rows)

subset, sampling_audit = balanced_sample_by_source_label(pool, N_PER_SOURCE_LABEL, seed=SEED)
subset = subset.sample(frac=1, random_state=SEED).reset_index(drop=True)

strata = subset['source'].astype(str) + '__' + subset['label'].astype(str)
train_val, test = train_test_split(subset, test_size=0.10, stratify=strata, random_state=SEED)
train_val_strata = train_val['source'].astype(str) + '__' + train_val['label'].astype(str)
train, val = train_test_split(train_val, test_size=0.111, stratify=train_val_strata, random_state=SEED)

train = train.reset_index(drop=True)
val = val.reset_index(drop=True)
test = test.reset_index(drop=True)

split_frames = {'train': train, 'val': val, 'test': test}

leakage_rows = []
for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    da = split_frames[a]
    db = split_frames[b]
    id_overlap = set(da['id']) & set(db['id'])
    title_overlap = (set(da['title_norm_split']) - {''}) & (set(db['title_norm_split']) - {''})
    leakage_rows.append({
        'split_pair': f'{a}-{b}',
        'id_overlap': len(id_overlap),
        'normalized_title_overlap': len(title_overlap),
        'title_overlap_examples': json.dumps(sorted(list(title_overlap))[:10]),
    })

split_leakage_audit = pd.DataFrame(leakage_rows)
if split_leakage_audit[['id_overlap', 'normalized_title_overlap']].to_numpy().sum() > 0:
    raise ValueError('Split leakage detected. Inspect split_leakage_audit before proceeding.')

split_summary = []
for split_name, df in split_frames.items():
    split_summary.append({
        'split': split_name,
        'n_rows': len(df),
        'fake': int((df['label'] == 'fake').sum()),
        'real': int((df['label'] == 'real').sum()),
        'politifact': int((df['source'] == 'politifact').sum()),
        'gossipcop': int((df['source'] == 'gossipcop').sum()),
    })
split_summary = pd.DataFrame(split_summary)

if WRITE_OUTPUTS:
    safe_to_csv(train, DATA_SPLIT_DIR / 'train.csv', index=False)
    safe_to_csv(val, DATA_SPLIT_DIR / 'val.csv', index=False)
    safe_to_csv(test, DATA_SPLIT_DIR / 'test.csv', index=False)
    safe_to_csv(sampling_audit, FINAL_ARTIFACT_DIR / 'cleaning_balanced_sampling_audit.csv', index=False)
    safe_to_csv(split_leakage_audit, FINAL_ARTIFACT_DIR / 'cleaning_id_title_leakage_audit.csv', index=False)
    safe_to_csv(split_summary, FINAL_ARTIFACT_DIR / 'cleaning_split_summary.csv', index=False)

display(sampling_audit)
display(split_leakage_audit)
display(split_summary)

## 5. Optional Valid-Image-Only Cleaning

In [ ]:
def load_split_for_image_filter(split):
    # Prefer the most current final split files. Older *_multimodal.csv files can
    # come from earlier split definitions, so they are only used as a fallback.
    candidates = [
        DATA_SPLIT_DIR / f'{split}_valid_images_only_image_deleaked_with_content.csv',
        DATA_SPLIT_DIR / f'{split}_valid_images_only_image_deleaked.csv',
        DATA_SPLIT_DIR / f'{split}_valid_images_only.csv',
        DATA_SPLIT_DIR / f'{split}_multimodal.csv',
        DATA_SPLIT_DIR / f'{split}.csv',
    ]
    for path in candidates:
        if path.exists():
            df = pd.read_csv(path)
            df['split'] = split
            return df, path
    raise FileNotFoundError(f'No split file found for {split}. Checked: {candidates}')

def as_bool_series(series):
    if series.dtype == bool:
        return series.fillna(False)
    return series.astype(str).str.lower().isin(['true', '1', 'yes'])

def attach_valid_image_flag(df):
    out = df.copy()
    path_col = 'image_path_primary' if 'image_path_primary' in out.columns else ('image_path' if 'image_path' in out.columns else None)
    if 'valid_image_path' in out.columns:
        out['valid_image_path'] = as_bool_series(out['valid_image_path'])
    elif path_col:
        out['valid_image_path'] = out[path_col].fillna('').astype(str).apply(lambda p: bool(p) and Path(p).exists())
    elif 'has_image' in out.columns:
        out['valid_image_path'] = as_bool_series(out['has_image'])
    else:
        out['valid_image_path'] = False
    return out

image_filter_rows = []
valid_image_splits = {}

for split in ['train', 'val', 'test']:
    df, loaded_from = load_split_for_image_filter(split)
    df = attach_valid_image_flag(df)
    valid_df = df[df['valid_image_path'].astype(bool)].reset_index(drop=True)
    valid_image_splits[split] = valid_df
    image_filter_rows.append({
        'split': split,
        'loaded_from': str(loaded_from),
        'rows_before': len(df),
        'rows_after_valid_image_filter': len(valid_df),
        'valid_image_coverage_%': round(100 * len(valid_df) / max(1, len(df)), 2),
    })
    if WRITE_OUTPUTS:
        safe_to_csv(valid_df, DATA_SPLIT_DIR / f'{split}_valid_images_only.csv', index=False)

valid_image_audit = pd.DataFrame(image_filter_rows)
if WRITE_OUTPUTS:
    safe_to_csv(valid_image_audit, FINAL_ARTIFACT_DIR / 'cleaning_valid_image_only_audit.csv', index=False)

display(valid_image_audit)

## 6. Optional Image-Content Leakage Audit and De-Leaked Valid-Image Splits

In [ ]:
def primary_image_path(row):
    for col in ['image_path_primary', 'image_path']:
        if col in row.index and isinstance(row[col], str) and row[col].strip():
            return row[col].strip()
    return ''

def sha256_file(path):
    path = Path(path)
    if not path.exists():
        return pd.NA
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

if not RUN_IMAGE_HASH_DELEAKAGE:
    print('Image hash de-leakage disabled by RUN_IMAGE_HASH_DELEAKAGE=False.')
else:
    hash_rows = []
    for split, df in valid_image_splits.items():
        for row in df.itertuples(index=False):
            row_s = pd.Series(row._asdict())
            path = primary_image_path(row_s)
            hash_rows.append({
                'split': split,
                'id': str(row_s.get('id', '')),
                'label': row_s.get('label', pd.NA),
                'source': row_s.get('source', pd.NA),
                'title': row_s.get('title', pd.NA),
                'image_path_primary': path,
                'image_sha256': sha256_file(path) if path else pd.NA,
            })

    image_hash_df = pd.DataFrame(hash_rows)
    if WRITE_OUTPUTS:
        safe_to_csv(image_hash_df, FINAL_ARTIFACT_DIR / 'cleaning_image_hash_manifest.csv', index=False)

    pair_rows = []
    for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
        ha = set(image_hash_df.loc[image_hash_df['split'].eq(a), 'image_sha256'].dropna())
        hb = set(image_hash_df.loc[image_hash_df['split'].eq(b), 'image_sha256'].dropna())
        overlap = ha & hb
        pair_rows.append({
            'split_pair': f'{a}-{b}',
            'n_overlapping_image_hashes': len(overlap),
            'overlap_%_of_smaller_split': round(100 * len(overlap) / max(1, min(len(ha), len(hb))), 3),
            'example_hashes': json.dumps(sorted(list(overlap))[:10]),
        })

    image_leakage_pair_summary = pd.DataFrame(pair_rows)
    if WRITE_OUTPUTS:
        safe_to_csv(image_leakage_pair_summary, FINAL_ARTIFACT_DIR / 'cleaning_image_hash_leakage_pair_summary.csv', index=False)

    def attach_hashes(df, split_name):
        out = df.copy()
        out['split'] = split_name
        existing_hash = (
            out['image_sha256'].reset_index(drop=True)
            if 'image_sha256' in out.columns
            else pd.Series([pd.NA] * len(out))
        )
        out = out.drop(columns=['image_sha256'], errors='ignore')
        key = image_hash_df[['split', 'id', 'image_sha256']].drop_duplicates(subset=['split', 'id'])
        out = out.merge(key, on=['split', 'id'], how='left').reset_index(drop=True)
        out['image_sha256'] = out['image_sha256'].combine_first(existing_hash)
        return out

    train_valid = attach_hashes(valid_image_splits['train'], 'train')
    val_valid = attach_hashes(valid_image_splits['val'], 'val')
    test_valid = attach_hashes(valid_image_splits['test'], 'test')

    train_hashes = set(train_valid['image_sha256'].dropna())
    val_keep = ~val_valid['image_sha256'].isin(train_hashes)
    val_deleaked = val_valid[val_keep].reset_index(drop=True)
    removed_val = val_valid[~val_keep].reset_index(drop=True)

    prior_hashes_for_test = train_hashes | set(val_deleaked['image_sha256'].dropna())
    test_keep = ~test_valid['image_sha256'].isin(prior_hashes_for_test)
    test_deleaked = test_valid[test_keep].reset_index(drop=True)
    removed_test = test_valid[~test_keep].reset_index(drop=True)

    image_deleakage_audit = pd.DataFrame([
        {'split': 'train', 'rows_before': len(train_valid), 'rows_after': len(train_valid), 'rows_removed_for_prior_split_image_overlap': 0},
        {'split': 'val', 'rows_before': len(val_valid), 'rows_after': len(val_deleaked), 'rows_removed_for_prior_split_image_overlap': len(removed_val)},
        {'split': 'test', 'rows_before': len(test_valid), 'rows_after': len(test_deleaked), 'rows_removed_for_prior_split_image_overlap': len(removed_test)},
    ])

    if WRITE_OUTPUTS:
        safe_to_csv(train_valid, DATA_SPLIT_DIR / 'train_valid_images_only_image_deleaked.csv', index=False)
        safe_to_csv(val_deleaked, DATA_SPLIT_DIR / 'val_valid_images_only_image_deleaked.csv', index=False)
        safe_to_csv(test_deleaked, DATA_SPLIT_DIR / 'test_valid_images_only_image_deleaked.csv', index=False)
        safe_to_csv(removed_val, FINAL_ARTIFACT_DIR / 'cleaning_removed_val_image_leakage_rows.csv', index=False)
        safe_to_csv(removed_test, FINAL_ARTIFACT_DIR / 'cleaning_removed_test_image_leakage_rows.csv', index=False)
        safe_to_csv(image_deleakage_audit, FINAL_ARTIFACT_DIR / 'cleaning_image_deleakage_audit.csv', index=False)

    display(image_leakage_pair_summary)
    display(image_deleakage_audit)
    if image_leakage_pair_summary['n_overlapping_image_hashes'].sum() == 0:
        print('No cross-split image-content leakage detected by SHA-256 hash.')

## 7. Cleaning Output Inventory

In [ ]:
expected_outputs = [
    DATA_SPLIT_DIR / 'clean_full.csv',
    DATA_SPLIT_DIR / 'train.csv',
    DATA_SPLIT_DIR / 'val.csv',
    DATA_SPLIT_DIR / 'test.csv',
    DATA_SPLIT_DIR / 'train_valid_images_only.csv',
    DATA_SPLIT_DIR / 'val_valid_images_only.csv',
    DATA_SPLIT_DIR / 'test_valid_images_only.csv',
    DATA_SPLIT_DIR / 'train_valid_images_only_image_deleaked.csv',
    DATA_SPLIT_DIR / 'val_valid_images_only_image_deleaked.csv',
    DATA_SPLIT_DIR / 'test_valid_images_only_image_deleaked.csv',
    FINAL_ARTIFACT_DIR / 'cleaning_row_count_audit.csv',
    FINAL_ARTIFACT_DIR / 'cleaning_id_title_leakage_audit.csv',
    FINAL_ARTIFACT_DIR / 'cleaning_valid_image_only_audit.csv',
    FINAL_ARTIFACT_DIR / 'cleaning_image_deleakage_audit.csv',
]

inventory = pd.DataFrame([
    {
        'path': str(path),
        'exists': path.exists(),
        'size_kb': round(path.stat().st_size / 1024, 1) if path.exists() else 0.0,
    }
    for path in expected_outputs
])
display(inventory)

print('Main cleaned split files are ready for downstream notebooks.')

if 'CSV_WRITE_LOG' in globals() and CSV_WRITE_LOG:
    print('CSV write log:')
    display(pd.DataFrame(CSV_WRITE_LOG))